# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nurana100/flyrank-ml-starter-nurana/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The rule, in plain words:** a page is worth reviewing first if it's stuck in the *91–180
day* no-update window (the window the signal check below shows is actually riskiest — not
just "the oldest" pages), it still has real search demand (≥ 500 impressions in 90 days), and
its click-through rate is below the median CTR for pages at its own position tier. That last
part matters: a page ranking on page 1 with a page-3 CTR isn't a ranking problem, it's a
**fixable snippet problem** (title/meta/schema) — exactly what the CTR-fix logic from the
session flags.

**Reason code (one, constant):** `aging_page_underperforming_ctr` when the rule fires,
`not_flagged` otherwise.

**Action label:** `refresh_and_fix_snippet` when flagged, `monitor` otherwise.

Before coding it, I check the two signals it leans on:

1. **Staleness → decline** (behind the refresh flags from the session): does "not updated in
   a while" actually track with `trend_direction == down`?
2. **CTR vs. position** (behind the CTR-fix logic from the session): does CTR really fall as
   position gets worse, the way the session's CTR-fix flag assumes?


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 120)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df):,} rows, {df['client_id'].nunique()} clients")

# ---------------------------------------------------------------------------
# Signal check 1 — staleness -> decline rate (behind the refresh flags)
# Claim: "the longer since a page was updated, the more likely it is declining."
# ---------------------------------------------------------------------------
tier_order = ["0-30", "31-90", "91-180", "181+"]
sig1 = (
    df.groupby("freshness_tier")
      .agg(n=("content_id", "size"),
           decline_rate=("trend_direction", lambda s: (s == "down").mean()))
      .reindex(tier_order)
)
sig1["decline_rate"] = sig1["decline_rate"].round(3)
overall_decline = round((df["trend_direction"] == "down").mean(), 3)

print("\nSignal 1 — freshness_tier vs. decline rate (n shown per bucket)")
print(sig1)
print(f"overall decline rate = {overall_decline}  (n={len(df):,})")

# Sample-size floor per the auditing-signals skill: no verdict under ~50 rows.
below_floor = sig1[sig1["n"] < 50]
verdict_1 = "MIXED"
print(f"\nVerdict: {verdict_1}")
print(
    "The story is NOT monotonic: 91-180 days is the riskiest bucket (0.611, n=9,171 — a real, "
    "well-powered read), but 181+ days (the most stale) actually comes in BELOW the overall "
    "rate (0.471), and that bucket's n=174 is thin enough that the gap could be noise (a rough "
    "SE at p~0.5, n=174 is ~3.8pp, so a 7pp drop is only borderline). 'Older = riskier' fails "
    "as a straight line. What DOES hold up on a large bucket: the 91-180 window specifically. "
    "That's the finding the rule below uses -- not a generic staleness cutoff."
)

# ---------------------------------------------------------------------------
# Signal check 2 — CTR vs. position (behind the CTR-fix logic) — flag-linked
# Claim: "CTR falls as position gets worse."
# ---------------------------------------------------------------------------
pos_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
sig2 = (
    df.groupby("position_tier")
      .apply(lambda d: pd.Series({
          "n": len(d),
          "median_impressions_90d": d["impressions_90d"].median(),
          "weighted_ctr_pct": (d["clicks_90d"].sum() / d["impressions_90d"].sum() * 100)
                               if d["impressions_90d"].sum() else np.nan,
      }))
      .reindex([t for t in pos_order if t in df["position_tier"].unique()])
)
sig2["n"] = sig2["n"].astype(int)
sig2 = sig2.round(3)

print("\nSignal 2 — position_tier vs. weighted CTR (n and volume floor shown per bucket)")
print(sig2)

verdict_2 = "CONFIRMED"
print(f"\nVerdict: {verdict_2}")
print(
    "CTR drops in step with position on every large bucket: page_1 0.350% -> striking 0.347% "
    "(basically tied) -> page_3_5 0.155% -> deep 0.041%, all n > 1,300. top_3 shows the highest "
    "CTR (0.488%) but per the data dictionary's volume-floor warning its median volume here is "
    "only 3 impressions/90d -- too thin to read on its own, so I lean on the large, well-powered "
    "buckets for the verdict. Confirmed on those: worse position -> lower CTR, as the CTR-fix "
    "logic assumes."
)


Loaded 30,000 rows, 32 clients

Signal 1 — freshness_tier vs. decline rate (n shown per bucket)
                    n  decline_rate
freshness_tier                     
0-30            20480         0.511
31-90             175         0.589
91-180           9171         0.611
181+              174         0.471
overall decline rate = 0.542  (n=30,000)

Verdict: MIXED
The story is NOT monotonic: 91-180 days is the riskiest bucket (0.611, n=9,171 — a real, well-powered read), but 181+ days (the most stale) actually comes in BELOW the overall rate (0.471), and that bucket's n=174 is thin enough that the gap could be noise (a rough SE at p~0.5, n=174 is ~3.8pp, so a 7pp drop is only borderline). 'Older = riskier' fails as a straight line. What DOES hold up on a large bucket: the 91-180 window specifically. That's the finding the rule below uses -- not a generic staleness cutoff.

Signal 2 — position_tier vs. weighted CTR (n and volume floor shown per bucket)
                   n  median_imp

## 2. Build the ranked queue (writes the CSV)

Code the rule above as a transparent, multiply-together score (no fitted weights) that leans
on both signals just checked: the 91-180 day freshness window, and a below-tier-median CTR.
Rank everything, attach the one reason code and one action label, write
`work/outputs/baseline_action_score.csv`.


In [2]:
from pathlib import Path

# --- the rule, as a readable score (multiply simple conditions -- no fitted weights) -------
aging_window   = (df["freshness_tier"] == "91-180").astype(int)          # signal 1 finding
visible        = (df["impressions_90d"] >= 500).astype(int)              # real demand floor
has_position   = (df["avg_position"] > 0).astype(int)                    # 0 = "no data", not rank 0
tier_median_ctr = df.groupby("position_tier")["ctr"].transform("median")
low_ctr_for_tier = ((df["ctr"] < tier_median_ctr) & (df["avg_position"] > 0)).astype(int)  # signal 2 finding

flag = aging_window * visible * has_position * low_ctr_for_tier          # readable on purpose

queue = df.copy()
queue["flag"] = flag
queue["baseline_action_score"] = flag * queue["impressions_90d"]
queue["reason_code"] = np.where(flag == 1, "aging_page_underperforming_ctr", "not_flagged")
queue["suggested_action"] = np.where(flag == 1, "refresh_and_fix_snippet", "monitor")
queue["baseline_rank"] = queue["baseline_action_score"].rank(method="first", ascending=False).astype(int)

queue = queue.sort_values("baseline_rank")

print(f"Flagged: {flag.sum():,} / {len(df):,} rows ({flag.mean()*100:.2f}%)")

decline_flagged = (queue.loc[queue['flag'] == 1, 'trend_direction'] == 'down').mean()
decline_overall = (queue['trend_direction'] == 'down').mean()
print(f"Decline rate among flagged rows: {decline_flagged:.3f}  (n={int(flag.sum())})")
print(f"Decline rate overall (base rate): {decline_overall:.3f}  (n={len(queue):,})")
print("-> the flagged group declines noticeably more often than the base rate. "
      "That's the honest signal this baseline is betting on -- and exactly what "
      "next week's model has to beat.")

output_columns = [
    "content_id", "client_id", "baseline_rank", "baseline_action_score", "flag",
    "reason_code", "suggested_action",
    "impressions_90d", "clicks_90d", "ctr", "avg_position", "position_tier",
    "days_since_last_update", "freshness_tier", "content_type", "main_intent",
    "search_volume", "trend_direction",
]

out_path = Path("../outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
queue[output_columns].to_csv(out_path, index=False)
print(f"\nWrote {len(queue):,} ranked rows -> {out_path.resolve().relative_to(Path.cwd().parent.parent)}")

# --- metrics receipt (this one IS committed -- see work/README.md) -------------------------
import json as _json
metrics = {
    "rows_total": int(len(queue)),
    "rows_flagged": int(flag.sum()),
    "flagged_share": round(float(flag.mean()), 4),
    "decline_rate_flagged": round(float(decline_flagged), 4),
    "decline_rate_overall": round(float(decline_overall), 4),
    "signal_1_verdict": "MIXED (91-180 day window elevated; monotonic staleness story does not hold)",
    "signal_2_verdict": "CONFIRMED (CTR falls as position tier worsens, on large buckets)",
    "rule": "freshness_tier==91-180 AND impressions_90d>=500 AND has_position AND ctr < tier-median ctr",
    "reason_code": "aging_page_underperforming_ctr",
    "action_label": "refresh_and_fix_snippet",
}
metrics_path = Path("../outputs/baseline_metrics.json")
metrics_path.write_text(_json.dumps(metrics, indent=2))
print(f"Wrote metrics -> {metrics_path.resolve().relative_to(Path.cwd().parent.parent)}")


Flagged: 2,016 / 30,000 rows (6.72%)
Decline rate among flagged rows: 0.706  (n=2016)
Decline rate overall (base rate): 0.542  (n=30,000)
-> the flagged group declines noticeably more often than the base rate. That's the honest signal this baseline is betting on -- and exactly what next week's model has to beat.



Wrote 30,000 ranked rows -> work/outputs/baseline_action_score.csv
Wrote metrics -> work/outputs/baseline_metrics.json


## 3. Top-10 review

*For each of the top 10: the action, why it's there, and what would make it wrong.*


In [3]:
top10 = queue.head(10).reset_index(drop=True)
cols = ["content_id", "client_id", "impressions_90d", "clicks_90d", "ctr", "avg_position",
        "position_tier", "days_since_last_update", "trend_direction", "search_volume",
        "main_intent", "content_type"]
print(top10[cols].to_string(index=False))

print("\n" + "=" * 100)
for i, row in top10.iterrows():
    why = (
        f"{row['position_tier']}-tier position (avg pos {row['avg_position']:.1f}) with {row['impressions_90d']:,} "
        f"impressions/90d but only {row['ctr']:.2f}% CTR ({row['clicks_90d']} clicks) -- well below "
        f"its tier's median, and it's sat {row['days_since_last_update']} days since the last update, "
        f"inside the 91-180 window signal 1 flagged as elevated-risk. trend_direction={row['trend_direction']!r}."
    )
    if row["search_volume"] == 0 or pd.isna(row["search_volume"]):
        wrong = "wrong if the keyword has no real commercial/informational value -- search_volume reads 0 here, so this may be ranking for a query nobody searches, in which case a snippet fix won't move clicks."
    elif row["trend_direction"] in ("stable", "up"):
        wrong = "wrong if the flat/rising trend means the low CTR is already priced in (e.g. a reference page people skim the SERP snippet for and don't need to click) rather than a fixable title/meta problem."
    else:
        wrong = "wrong if the CTR drop traces to a SERP feature change (featured snippet stealing clicks, a new competitor) rather than to our own title/meta -- a snippet fix wouldn't help that."
    print(f"#{i+1}  {row['content_id']}")
    print(f"  action: {row['suggested_action']}  (reason_code={row['reason_code']})")
    print(f"  why:    {why}")
    print(f"  what would make it wrong: {wrong}")
    print()


          content_id         client_id  impressions_90d  clicks_90d  ctr  avg_position position_tier  days_since_last_update trend_direction  search_volume   main_intent    content_type
content_5fe46e04994d client_4e07408562           517715         741 0.14           4.2        page_1                     104            down         1900.0 informational keyword article
content_36ff89c8214e client_19581e27de           295097         154 0.05           7.3        page_1                     104          stable            0.0 informational keyword article
content_c8e9d6ab9013 client_19581e27de           208678           0 0.00           9.7        page_1                     104            down           20.0 informational keyword article
content_a7427266c305 client_19581e27de           201111         219 0.11           5.7        page_1                     104          stable            0.0 informational keyword article
content_91652435f57a client_19581e27de           159590         100 0.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no future windows or label-derived columns leaked in.*


In [4]:
# --- weak pick: client concentration in the top 10 -----------------------------------------
client_counts = top10["client_id"].value_counts()
print("Top-10 rows by client:")
print(client_counts)
dominant_share = client_counts.iloc[0] / len(top10)
print(f"\nWeak pick: {client_counts.iloc[0]} of the top 10 ({dominant_share:.0%}) belong to a single "
      f"client ({client_counts.index[0]}). The score has no per-client normalization, so one client "
      "with many aging, high-traffic pages can crowd out equally-worth-reviewing pages at smaller "
      "clients. Fix for the model week: rank within client, or z-score the score per client, before "
      "taking a global top-K.")

# --- leakage check ---------------------------------------------------------------------------
score_inputs = {"freshness_tier", "impressions_90d", "avg_position", "ctr", "position_tier"}
forbidden = {"trend_direction", "trend_pct", "is_declining_label"}  # label source / label itself
print("\nColumns the score formula actually reads from:", sorted(score_inputs))
assert score_inputs.isdisjoint(forbidden), "leakage: a label-derived column was used in the score"
print("No overlap with forbidden label-source columns", forbidden, "-> PASS")

future_window_cols = {"impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
                       "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"}
print("No 30-day trend-window columns used in the score:",
      score_inputs.isdisjoint(future_window_cols), "-> PASS")

print("\ntrend_direction / trend_pct appear ONLY as read-only context for the signal checks and "
      "the human review above -- never inside the score, reason_code, or action_label logic.")


Top-10 rows by client:
client_id
client_19581e27de    8
client_4e07408562    1
client_6208ef0f77    1
Name: count, dtype: int64

Weak pick: 8 of the top 10 (80%) belong to a single client (client_19581e27de). The score has no per-client normalization, so one client with many aging, high-traffic pages can crowd out equally-worth-reviewing pages at smaller clients. Fix for the model week: rank within client, or z-score the score per client, before taking a global top-K.

Columns the score formula actually reads from: ['avg_position', 'ctr', 'freshness_tier', 'impressions_90d', 'position_tier']
No overlap with forbidden label-source columns {'trend_pct', 'is_declining_label', 'trend_direction'} -> PASS
No 30-day trend-window columns used in the score: True -> PASS

trend_direction / trend_pct appear ONLY as read-only context for the signal checks and the human review above -- never inside the score, reason_code, or action_label logic.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
